<a href="https://colab.research.google.com/github/madaan/llm-reasoning-tutorial/blob/main/Part_0_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 0: Foundations — From SGD to Transformers

This notebook builds the key concepts behind modern language models from scratch:

1. **Gradient Descent & SGD** — how neural networks learn
2. **Backpropagation** — computing gradients efficiently
3. **Word Embeddings** — representing words as vectors
4. **Attention Mechanism** — the core of transformers
5. **The Transformer Block** — putting it all together

Everything runs locally with PyTorch — no API keys needed.

In [ ]:
!pip install torch numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

---
## 1. Gradient Descent & SGD

All neural network training boils down to:
1. Define a **loss function** that measures how wrong the model is
2. Compute the **gradient** of the loss with respect to each parameter
3. **Update** parameters in the direction that reduces the loss

$$\theta_{t+1} = \theta_t - \eta \cdot \nabla_{\theta} \mathcal{L}(\theta_t)$$

where $\eta$ is the learning rate.

Let's see this in action with a simple example: fitting a line $y = wx + b$.

In [ ]:
# Generate synthetic data: y = 3x + 1 + noise
true_w, true_b = 3.0, 1.0
X = torch.randn(100, 1)
y = true_w * X + true_b + 0.1 * torch.randn(100, 1)

# Initialize random parameters
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

learning_rate = 0.1
losses = []

for step in range(100):
    # Forward pass: compute prediction and loss
    y_pred = w * X + b
    loss = ((y_pred - y) ** 2).mean()  # Mean Squared Error
    losses.append(loss.item())

    # Backward pass: compute gradients
    loss.backward()

    # Update parameters (SGD step)
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        w.grad.zero_()
        b.grad.zero_()

print(f"Learned: w={w.item():.4f} (true: {true_w}), b={b.item():.4f} (true: {true_b})")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Loss over training steps")

plt.subplot(1, 2, 2)
plt.scatter(X.numpy(), y.numpy(), alpha=0.5, s=10, label="Data")
x_line = torch.linspace(-3, 3, 100).unsqueeze(1)
plt.plot(x_line.numpy(), (w * x_line + b).detach().numpy(), 'r-', linewidth=2, label="Learned")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Data and learned fit")
plt.tight_layout()
plt.show()

### Interactive: Effect of Learning Rate

The learning rate $\eta$ is crucial. Too small → slow convergence. Too large → divergence.

In [ ]:
def train_with_lr(lr, steps=50):
    w = torch.randn(1, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    losses = []
    for _ in range(steps):
        y_pred = w * X + b
        loss = ((y_pred - y) ** 2).mean()
        losses.append(loss.item())
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
    return losses

plt.figure(figsize=(10, 5))
for lr in [0.001, 0.01, 0.1, 0.5]:
    losses = train_with_lr(lr)
    plt.plot(losses, label=f"lr={lr}")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Effect of Learning Rate on Convergence")
plt.legend()
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

### Stochastic vs. Batch Gradient Descent

- **Batch GD**: Use all data to compute gradient → accurate but slow
- **SGD**: Use a random mini-batch → noisy but fast, helps escape local minima
- **Mini-batch SGD**: Use a small batch (e.g., 32 samples) → best of both worlds

In [ ]:
def train_sgd(batch_size, steps=100):
    w = torch.tensor([0.0], requires_grad=True)
    b = torch.tensor([0.0], requires_grad=True)
    losses = []
    for step in range(steps):
        # Sample mini-batch
        idx = torch.randint(0, len(X), (batch_size,))
        X_batch, y_batch = X[idx], y[idx]

        y_pred = w * X_batch + b
        loss = ((y_pred - y_batch) ** 2).mean()

        # Track full-batch loss for comparison
        with torch.no_grad():
            full_loss = ((w * X + b - y) ** 2).mean()
        losses.append(full_loss.item())

        loss.backward()
        with torch.no_grad():
            w -= 0.1 * w.grad
            b -= 0.1 * b.grad
            w.grad.zero_()
            b.grad.zero_()
    return losses

plt.figure(figsize=(10, 5))
for bs in [1, 8, 32, 100]:
    label = f"batch_size={bs}" + (" (full batch)" if bs == 100 else "")
    plt.plot(train_sgd(bs), label=label, alpha=0.8)
plt.xlabel("Step")
plt.ylabel("Loss (full data)")
plt.title("Stochastic vs Batch Gradient Descent")
plt.legend()
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

---
## 2. Backpropagation

Backpropagation is the algorithm that computes gradients efficiently using the **chain rule**.

For a composition of functions $f(g(x))$:
$$\frac{\partial f}{\partial x} = \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$$

Let's trace through a tiny neural network **by hand** and verify with PyTorch autograd.

In [ ]:
# A tiny 2-layer network: input -> linear -> ReLU -> linear -> loss
# x (input) -> h = w1*x + b1 -> a = ReLU(h) -> y_pred = w2*a + b2 -> loss = (y_pred - y_true)^2

x_val = torch.tensor([2.0])
y_true = torch.tensor([1.0])

w1 = torch.tensor([0.5], requires_grad=True)
b1 = torch.tensor([0.1], requires_grad=True)
w2 = torch.tensor([-0.3], requires_grad=True)
b2 = torch.tensor([0.2], requires_grad=True)

# Forward pass (step by step)
h = w1 * x_val + b1          # h = 0.5*2 + 0.1 = 1.1
a = torch.relu(h)              # a = ReLU(1.1) = 1.1
y_pred = w2 * a + b2           # y_pred = -0.3*1.1 + 0.2 = -0.13
loss = (y_pred - y_true) ** 2  # loss = (-0.13 - 1)^2 = 1.2769

print("=== Forward Pass ===")
print(f"h = {h.item():.4f}")
print(f"a = ReLU(h) = {a.item():.4f}")
print(f"y_pred = {y_pred.item():.4f}")
print(f"loss = {loss.item():.4f}")

# Backward pass
loss.backward()

print("\n=== Backward Pass (Gradients) ===")
print(f"d(loss)/d(w2) = {w2.grad.item():.4f}")
print(f"d(loss)/d(b2) = {b2.grad.item():.4f}")
print(f"d(loss)/d(w1) = {w1.grad.item():.4f}")
print(f"d(loss)/d(b1) = {b1.grad.item():.4f}")

# Manual verification
print("\n=== Manual Verification ===")
dl_dy = 2 * (y_pred.item() - y_true.item())  # d(loss)/d(y_pred)
dy_dw2 = a.item()                              # d(y_pred)/d(w2) = a
dy_db2 = 1.0                                   # d(y_pred)/d(b2) = 1
print(f"d(loss)/d(w2) = {dl_dy * dy_dw2:.4f} (manual)")
print(f"d(loss)/d(b2) = {dl_dy * dy_db2:.4f} (manual)")

### Training a Small Neural Network

Let's put SGD + backprop together to train a 2-layer neural network on a nonlinear function.

In [ ]:
# Generate nonlinear data: y = sin(x) + noise
X_nl = torch.linspace(-3, 3, 200).unsqueeze(1)
y_nl = torch.sin(X_nl) + 0.1 * torch.randn_like(X_nl)

# Define a simple 2-layer neural network
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 1)
)

optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
losses = []

for epoch in range(500):
    y_pred = model(X_nl)
    loss = F.mse_loss(y_pred, y_nl)
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")

plt.subplot(1, 2, 2)
with torch.no_grad():
    y_fit = model(X_nl)
plt.scatter(X_nl.numpy(), y_nl.numpy(), alpha=0.3, s=5, label="Data")
plt.plot(X_nl.numpy(), y_fit.numpy(), 'r-', linewidth=2, label="Neural Net")
plt.plot(X_nl.numpy(), torch.sin(X_nl).numpy(), 'g--', linewidth=1, label="True sin(x)")
plt.legend()
plt.title("Neural Network fits sin(x)")
plt.tight_layout()
plt.show()

---
## 3. Word Embeddings

Neural networks work with numbers, not words. **Embeddings** map each word (or token) to a dense vector.

Key idea: words with similar meanings should have similar vectors.

$$\text{embed}(\text{"king"}) - \text{embed}(\text{"man"}) + \text{embed}(\text{"woman"}) \approx \text{embed}(\text{"queen"})$$

In [ ]:
# Simple vocabulary and embedding
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "in", "park", "house"]
vocab_size = len(vocab)
embed_dim = 8

# Create an embedding layer
embedding = nn.Embedding(vocab_size, embed_dim)

# Map words to indices
word_to_idx = {w: i for i, w in enumerate(vocab)}

# Embed a sentence
sentence = ["the", "cat", "sat", "on", "the", "mat"]
indices = torch.tensor([word_to_idx[w] for w in sentence])
embedded = embedding(indices)

print(f"Sentence: {sentence}")
print(f"Indices:  {indices.tolist()}")
print(f"Embedded shape: {embedded.shape}  (seq_len={len(sentence)}, embed_dim={embed_dim})")
print(f"\nEmbedding for 'cat':\n{embedded[1].detach().numpy().round(3)}")

# Note: same word ('the') gets same embedding
print(f"\n'the' at position 0 == 'the' at position 4: {torch.equal(embedded[0], embedded[4])}")

### Positional Encoding

Transformers process all positions in parallel, so they need **positional encodings** to know word order.

The original transformer uses sinusoidal functions:
$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d})$$

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe = positional_encoding(50, 64)

plt.figure(figsize=(12, 4))
plt.imshow(pe.numpy().T, aspect='auto', cmap='RdBu')
plt.xlabel("Position in sequence")
plt.ylabel("Embedding dimension")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar()
plt.show()

# Show that nearby positions have similar encodings
print("Cosine similarity between adjacent positions:")
for i in range(0, 5):
    sim = F.cosine_similarity(pe[i].unsqueeze(0), pe[i+1].unsqueeze(0)).item()
    print(f"  pos {i} vs pos {i+1}: {sim:.4f}")
print(f"\nCosine similarity between distant positions:")
sim = F.cosine_similarity(pe[0].unsqueeze(0), pe[25].unsqueeze(0)).item()
print(f"  pos 0 vs pos 25: {sim:.4f}")

---
## 4. Attention Mechanism

Attention is the core innovation of the Transformer. It allows each token to "look at" every other token and decide what's relevant.

### Scaled Dot-Product Attention

Given queries $Q$, keys $K$, and values $V$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Intuition:
- **Query**: "What am I looking for?"
- **Key**: "What do I contain?"
- **Value**: "What information do I provide?"
- The dot product $QK^T$ measures compatibility; softmax turns it into weights.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Compute scaled dot-product attention."""
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


# Demo: attention on a short sentence
torch.manual_seed(42)
seq_len = 6  # "the cat sat on the mat"
d_model = 16

# Create random Q, K, V (in practice, these come from linear projections of embeddings)
Q = torch.randn(1, seq_len, d_model)
K = torch.randn(1, seq_len, d_model)
V = torch.randn(1, seq_len, d_model)

output, weights = scaled_dot_product_attention(Q, K, V)

words = ["the", "cat", "sat", "on", "the", "mat"]
plt.figure(figsize=(6, 5))
plt.imshow(weights[0].detach().numpy(), cmap='Blues')
plt.xticks(range(seq_len), words)
plt.yticks(range(seq_len), words)
plt.xlabel("Key (attending to)")
plt.ylabel("Query (attending from)")
plt.title("Attention Weights")
plt.colorbar()
plt.show()

print("Each row sums to 1 (softmax):")
print(weights[0].sum(dim=-1).detach().numpy().round(4))

### Causal (Autoregressive) Masking

For language models (like GPT), each token can only attend to **previous** tokens — not future ones. This is enforced with a causal mask.

In [ ]:
# Create causal mask: lower triangular matrix
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)

output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(causal_mask[0].numpy(), cmap='Greys')
axes[0].set_title("Causal Mask\n(1=allowed, 0=blocked)")
axes[0].set_xticks(range(seq_len))
axes[0].set_xticklabels(words)
axes[0].set_yticks(range(seq_len))
axes[0].set_yticklabels(words)

axes[1].imshow(weights[0].detach().numpy(), cmap='Blues')
axes[1].set_title("Without mask (bidirectional)")
axes[1].set_xticks(range(seq_len))
axes[1].set_xticklabels(words)
axes[1].set_yticks(range(seq_len))
axes[1].set_yticklabels(words)

axes[2].imshow(weights_masked[0].detach().numpy(), cmap='Blues')
axes[2].set_title("With causal mask (autoregressive)")
axes[2].set_xticks(range(seq_len))
axes[2].set_xticklabels(words)
axes[2].set_yticks(range(seq_len))
axes[2].set_yticklabels(words)

plt.tight_layout()
plt.show()

print("With causal mask, 'sat' (position 2) can only attend to 'the', 'cat', 'sat':")
print(f"  Weights: {weights_masked[0, 2].detach().numpy().round(3)}")

### Multi-Head Attention

Instead of one attention function, transformers use **multiple heads** in parallel. Each head can learn different types of relationships (e.g., one head for syntax, another for semantics).

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        # Project to Q, K, V and split into heads
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention for each head
        output, weights = scaled_dot_product_attention(Q, K, V, mask)

        # Concatenate heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        output = self.W_o(output)
        return output, weights


# Demo
mha = MultiHeadAttention(d_model=16, num_heads=4)
x = torch.randn(1, 6, 16)  # batch=1, seq_len=6, d_model=16
out, attn = mha(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Attention weights shape: {attn.shape}  (batch, heads, seq, seq)")

# Visualize each head's attention pattern
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for h in range(4):
    axes[h].imshow(attn[0, h].detach().numpy(), cmap='Blues')
    axes[h].set_title(f"Head {h+1}")
    axes[h].set_xticks(range(6))
    axes[h].set_xticklabels(words, fontsize=8)
    axes[h].set_yticks(range(6))
    axes[h].set_yticklabels(words, fontsize=8)
plt.suptitle("Different Heads Learn Different Attention Patterns", fontsize=12)
plt.tight_layout()
plt.show()

---
## 5. The Transformer Block

A transformer block combines:
1. **Multi-head self-attention** (with residual connection and layer norm)
2. **Feed-forward network** (with residual connection and layer norm)

```
x → LayerNorm → Multi-Head Attention → + (residual) → LayerNorm → FFN → + (residual) → output
```

GPT-2, for example, stacks 12 of these blocks (for the small version).

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention with residual
        normed = self.norm1(x)
        attn_out, attn_weights = self.attention(normed, mask)
        x = x + self.dropout(attn_out)

        # Feed-forward with residual
        normed = self.norm2(x)
        ff_out = self.ffn(normed)
        x = x + self.dropout(ff_out)

        return x, attn_weights


# A mini GPT-like model
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, d_ff=128, num_layers=2, max_len=128):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        tok_emb = self.token_embed(idx)
        pos_emb = self.pos_embed(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb

        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x, _ = block(x, mask)

        x = self.norm(x)
        logits = self.head(x)
        return logits


# Create and inspect
mini_gpt = MiniGPT(vocab_size=100)
total_params = sum(p.numel() for p in mini_gpt.parameters())
print(f"MiniGPT parameters: {total_params:,}")
print(f"\nFor reference:")
print(f"  GPT-2 small:  124M parameters")
print(f"  GPT-2 medium: 355M parameters")
print(f"  GPT-2 large:  774M parameters")
print(f"  GPT-3:        175B parameters")

# Test forward pass
test_input = torch.randint(0, 100, (1, 10))  # batch=1, seq_len=10
logits = mini_gpt(test_input)
print(f"\nInput shape:  {test_input.shape}")
print(f"Output shape: {logits.shape}  (batch, seq_len, vocab_size)")
print(f"Each position predicts a probability distribution over {logits.shape[-1]} tokens.")

### Training MiniGPT on a Toy Dataset

Let's train our mini transformer to learn simple patterns. We'll teach it to repeat sequences.

In [ ]:
# Toy task: learn to copy a sequence
# Input:  [A, B, C, SEP, ?, ?, ?]
# Target: [_, _, _, _, A, B, C]

vocab_size = 12  # 0-9 for digits, 10=SEP, 11=PAD
SEP = 10

def make_copy_data(num_samples=1000, seq_len=3):
    """Generate copy task data."""
    data = []
    for _ in range(num_samples):
        seq = torch.randint(0, 10, (seq_len,))
        # Input: seq + SEP + seq
        full_input = torch.cat([seq, torch.tensor([SEP]), seq])
        data.append(full_input)
    return torch.stack(data)

data = make_copy_data(2000, seq_len=3)
print(f"Data shape: {data.shape}")
print(f"Example: {data[0].tolist()}")
print(f"  Input part:  {data[0][:3].tolist()} SEP")
print(f"  Copy target: {data[0][4:].tolist()}")

# Train
model = MiniGPT(vocab_size=vocab_size, d_model=32, num_heads=4, d_ff=64, num_layers=2, max_len=10)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(200):
    # Mini-batch
    idx = torch.randint(0, len(data), (64,))
    batch = data[idx]

    # Predict next token at each position
    logits = model(batch[:, :-1])  # Input: all but last
    targets = batch[:, 1:]          # Target: all but first

    loss = F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))
    losses.append(loss.item())

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}: loss = {loss.item():.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy Loss")
plt.title("Training MiniGPT on Copy Task")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Test the trained model
model.eval()
test_data = make_copy_data(10, seq_len=3)

print("Testing copy task:")
print(f"{'Input':>20} | {'Expected':>10} | {'Predicted':>10} | {'Correct':>8}")
print("-" * 60)

correct = 0
for i in range(10):
    # Feed input up to SEP, then generate
    input_seq = test_data[i][:4]  # [A, B, C, SEP]
    generated = input_seq.tolist()

    for _ in range(3):
        with torch.no_grad():
            logits = model(torch.tensor([generated]))
            next_token = logits[0, -1].argmax().item()
            generated.append(next_token)

    expected = test_data[i][4:].tolist()
    predicted = generated[4:]
    is_correct = expected == predicted
    correct += is_correct

    print(f"{str(generated[:4]):>20} | {str(expected):>10} | {str(predicted):>10} | {'Y' if is_correct else 'N':>8}")

print(f"\nAccuracy: {correct}/10")

---
## Summary

We covered the key building blocks of modern language models:

| Concept | What it does | Why it matters |
|---------|-------------|----------------|
| **SGD** | Updates parameters using gradients | The core optimization algorithm |
| **Backpropagation** | Efficiently computes gradients | Makes training deep networks feasible |
| **Embeddings** | Maps tokens to vectors | Gives the model a numeric representation of language |
| **Positional Encoding** | Encodes word order | Lets the model know where each token is |
| **Self-Attention** | Tokens attend to each other | Captures long-range dependencies |
| **Multi-Head Attention** | Multiple attention patterns | Different heads capture different relationships |
| **Transformer Block** | Attention + FFN + residuals | The repeating unit in GPT, BERT, etc. |

### Next Steps
- **Part 1**: See how GPT-2 (a real transformer) estimates sentence probabilities and predicts next words
- **Part 2**: Learn prompting basics — how to communicate effectively with LLMs
- **Part 3**: Advanced prompting techniques like chain-of-thought reasoning